# Semana 12: Aprendizaje semi-supervisado (Pseudo-labeling)

## Impacto ambiental de la energia

Este notebook utiliza las etiquetas artificiales creadas por K-means en Semana 10 para entrenar clasificadores supervisados. Cada etiqueta representa un perfil energetico region-categoria, no una clase real observada.

## Flujo de trabajo

1. Recuperar el Parquet `datos_etiquetados_kmeans` guardado por Semana 10.
2. Renombrar el cluster K-means como `label` para pseudo-labeling.
3. Entrenar y evaluar Arbol de Decision, Random Forest, SVM OneVsRest y Regresion Logistica Multinomial.
4. Visualizar las reglas aproximadas del arbol y su matriz de confusion.
5. Entrenar una regresion lineal para predecir intensidad CO2 sin incluir emisiones como predictor directo.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

OUTPUT_DIR = Path('salidas')
PARQUET_PATH = Path('../Semana 10 Clustering/modelos/datos_etiquetados_kmeans')
assert PARQUET_PATH.exists(), 'Ejecuta primero Clustering_Semana10.ipynb para guardar las pseudo-etiquetas.'
print('Pseudo-etiquetas disponibles en:', PARQUET_PATH)

## Clasificacion supervisada a partir de pseudo-etiquetas

La clasificacion mide que tan bien distintos algoritmos pueden reproducir los grupos definidos por K-means. Un accuracy alto no significa que se haya descubierto una verdad causal: indica que las fronteras geometricas del clustering son aprendibles.

In [ ]:
%run ./pseudo_labeling_semana12.py

In [ ]:
clasificacion = pd.read_csv(OUTPUT_DIR / 'metricas_clasificacion.csv')
confusion = pd.read_csv(OUTPUT_DIR / 'matriz_confusion_arbol.csv', index_col=0)
display(clasificacion.style.format({'accuracy': '{:.2%}'}))
display(confusion)

## Visualizacion del arbol y comparacion de clasificadores

Spark imprime la estructura logica del arbol. Para obtener una figura legible, se entrena un arbol equivalente de Scikit-Learn sobre la pequena tabla analitica, exclusivamente con fines de visualizacion.

In [ ]:
for nombre in ['accuracy_modelos.png', 'matriz_confusion_arbol.png', 'arbol_pseudo_etiquetas.png']:
    display(Image(filename=str(OUTPUT_DIR / 'figuras' / nombre)))

## Regresion lineal: intensidad de emisiones

Para el dominio energetico, la variable continua elegida es `intensidad_co2` (toneladas CO2/MWh). Se predice solamente con `log_generacion` y `participacion_generacion`. No se usa `emisiones_co2_mt`, porque forma parte de la definicion de intensidad y produciria fuga de informacion.

In [ ]:
regresion = pd.read_csv(OUTPUT_DIR / 'metricas_regresion.csv')
display(regresion)
display(Image(filename=str(OUTPUT_DIR / 'figuras' / 'regresion_intensidad_co2.png')))

## Ticket de salida

1. Un alto accuracy en clasificacion se explica porque el modelo intenta reproducir etiquetas generadas por K-means a partir de las mismas variables: aprende una separacion geometrica, no necesariamente un fenomeno causal.
2. Para predecir impacto ambiental real sin hacer trampa se necesitan variables adicionales: tecnologia especifica, combustible utilizado, eficiencia de planta, factor de capacidad, controles de emisiones y condiciones operacionales.
3. El desempeno de la regresion debe interpretarse con cautela: generacion y participacion regional no explican por si solas toda la intensidad de CO2.